# 04 — Baselines (E0): qué miden y qué nos enseñó el SMOKE

**Fase 2, en retrospectiva.** Antes de entrenar nada hay que responder una pregunta incómoda: ¿qué tan bien lo hace el modelo base *sin* entrenar? Si no medimos eso primero, cualquier número posterior al fine-tuning es propaganda, no evidencia: no sabríamos si la mejora viene del entrenamiento o si el modelo ya sabía hacerlo.

Por eso existen los baselines **E0** del plan (PLAN.md §19/§34):

| ID | Qué es | Qué pregunta responde |
|---|---|---|
| **E0a** | Modelo base, **zero-shot**: solo el system prompt de §8 y la petición del usuario | ¿Cuánto sabe ya el modelo del formato Skill Blueprint? |
| **E0b** | Modelo base, **few-shot**: el mismo prompt + 3 ejemplos fijos del split de *train* (nunca de val/test) | ¿El fine-tuning aporta algo más allá de «enseñar el formato con ejemplos en el prompt»? |

Si E0b saliera casi tan bien como el modelo entrenado, el fine-tuning sería difícil de justificar. Los baselines son el grupo de control del experimento.

> **Honestidad de etiquetas:** lo que este notebook analiza es el **SMOKE** de la Fase 2 (`experiments/e0_smoke/`): Qwen2.5-**0.5B**-Instruct, zero-shot, solo 5 registros de val, en CPU. Es una prueba de que el *pipeline* funciona, **no** el resultado E0 real (7B, val/test completos), que sigue pendiente de la GPU de Kaggle.

## Cómo puntúa el harness: la escalera de 5 capas

La puntuación no la inventamos en el pipeline de baselines: `score_predictions` **importa y reutiliza** `scripts/eval_harness.py`, la misma escalera de PLAN.md §29:

| Capa | Qué comprueba | Peso (§33) |
|---|---|---|
| **L1** | ¿La salida es JSON parseable? | 0.25 |
| **L2** | ¿Cumple el schema del Skill Blueprint? (campos obligatorios, nombre kebab-case…) | 0.25 |
| **L3** | Precisión de campos vs. referencia: name exact-match, F1 de rutas de archivos, F1 de tokens de la descripción | 0.20 |
| **L4** | LLM-as-judge (juicio cualitativo) | 0.15 |
| **L5** | ¿El blueprint se puede *materializar* en archivos reales en un directorio temporal? | 0.15 |

**SkillGen Score** = 0.25·L1 + 0.25·L2 + 0.20·L3 + 0.15·L4 + 0.15·L5.

### El problema de L4 y la renormalización (léelo antes de citar el número)

L4 necesita un juez LLM con `OPENROUTER_API_KEY`. En el SMOKE **no se ejecutó** (`"l4_judge": "skipped"`). ¿Qué hacer con su peso (0.15) al calcular el score global? La decisión — la misma que toma el propio harness al omitir L4 de su reporte — es **renormalizar sobre las capas disponibles**: sumar los pesos de L1+L2+L3+L5 (0.25+0.25+0.20+0.15 = **0.85**) y dividir la suma ponderada entre 0.85. Así el score sigue viviendo en [0, 1] y un run perfecto sin juez puntuaría 1.0.

La contrapartida honesta: un score renormalizado **no es comparable** con un score con L4 incluido. Por eso `scores.json` guarda también `skillgen_score_weighted_sum_no_l4` (la suma sin dividir, máximo 0.85) y este notebook siempre dice «renormalizado, sin L4» cuando cita el 0.3245.

## Cargamos los resultados del SMOKE

Leemos los artefactos que dejó el run de la Fase 2 — **no regeneramos nada** (la generación tardó ~7 minutos en CPU; la última celda muestra el comando exacto para reproducirla).

In [1]:
import json
from pathlib import Path

SMOKE = Path("../experiments/e0_smoke")
scores = json.loads((SMOKE / "scores.json").read_text())
predictions = [json.loads(line) for line in (SMOKE / "predictions.jsonl").read_text().splitlines()]

print(f"Modelo: {predictions[0]['model']}  |  modo: {predictions[0]['mode']}  |  split: {predictions[0]['split']}")
print(f"n = {scores['n']} predicciones")
for clave in ["l1_parse_rate", "l2_schema_rate", "l3", "l5_materialize_rate"]:
    print(f"{clave:24s} {scores[clave]:.4f}")
print(f"{'l4_judge':24s} {scores['l4_judge']} (sin OPENROUTER_API_KEY)")
print(f"{'SkillGen Score':24s} {scores['skillgen_score']:.4f}  <- renormalizado sobre 0.85 (sin L4)")
print(f"{'suma ponderada (máx .85)':24s} {scores['skillgen_score_weighted_sum_no_l4']:.4f}")

Modelo: Qwen/Qwen2.5-0.5B-Instruct  |  modo: zero_shot  |  split: val
n = 5 predicciones
l1_parse_rate            1.0000
l2_schema_rate           0.0000
l3                       0.1290
l5_materialize_rate      0.0000
l4_judge                 skipped (sin OPENROUTER_API_KEY)
SkillGen Score           0.3245  <- renormalizado sobre 0.85 (sin L4)
suma ponderada (máx .85) 0.2758


Lectura rápida: **L1 = 1.0** (las 5 salidas son JSON válido y coherente) pero **L2 = 0, L5 = 0**. El modelo pequeño «sabe JSON», pero no conoce *nuestro* schema. Esa brecha — formato genérico vs. schema exacto — es exactamente lo que el fine-tuning tiene que cerrar. Vamos a verlo en un caso concreto.

## Anatomía de un fallo: predicción vs. referencia

Tomamos la primera predicción y su referencia real del split de validación (el split congelado; solo lo *leemos*).

In [2]:
# Referencia real (split val congelado) de la primera predicción
val_records = [json.loads(line) for line in Path("../data/splits/val.jsonl").read_text().splitlines()]
by_id = {r["metadata"]["id"]: r for r in val_records}

pred = predictions[0]
ref_blueprint = json.loads(by_id[pred["id"]]["expected_blueprint"])
pred_blueprint = json.loads(pred["output"].strip().strip("`").removeprefix("json"))

print("ID:", pred["id"])
print("\nClaves de la REFERENCIA:", sorted(ref_blueprint))
print("Claves de la PREDICCIÓN:", sorted(pred_blueprint))
print("\nname  referencia:", ref_blueprint["name"])
print("name  predicción:", pred_blueprint["name"], " <- Title/CamelCase, no kebab-case")
print("\nArchivos en la referencia:", len(ref_blueprint["files"]), "| ¿'files' en la predicción?", "files" in pred_blueprint)

ID: davila7_claude-code-templates__clinical-reports

Claves de la REFERENCIA: ['blueprint_version', 'description', 'files', 'name', 'purpose', 'when_not_to_use', 'when_to_use']
Claves de la PREDICCIÓN: ['description', 'name', 'outputFormat', 'parameters', 'requirements']

name  referencia: clinical-reports
name  predicción: ClinicalReportGenerator  <- Title/CamelCase, no kebab-case

Archivos en la referencia: 31 | ¿'files' en la predicción? False


In [3]:
# ¿Qué dijo exactamente la capa L2 de esta predicción?
item = next(i for i in scores["items"] if i["id"] == pred["id"])
print("Errores L2 (schema):")
for error in item["l2_errors"]:
    print(" -", error)
print("\nL3:", {k: round(v, 4) for k, v in item["l3"].items()})
print("L5:", item["l5_errors"])

Errores L2 (schema):
 - missing field: blueprint_version
 - missing field: purpose
 - missing field: when_to_use
 - missing field: when_not_to_use
 - missing field: files
 - invalid name: 'ClinicalReportGenerator'

L3: {'name_em': 0.0, 'file_path_f1': 0.0, 'desc_token_f1': 0.5862}
L5: ['SKILL.md not materialized']


### Qué nos dicen los errores L2: el schema inventado

Fíjate en el patrón, porque se repite en las 5 predicciones del SMOKE (todas tienen la misma lista de errores L2):

1. **Faltan los 5 campos estructurales**: `blueprint_version`, `purpose`, `when_to_use`, `when_not_to_use` y `files`. Sin `files`, L5 es imposible por definición: no hay nada que materializar (`"SKILL.md not materialized"`).
2. **El nombre es inválido**: `ClinicalReportGenerator` (CamelCase) en vez de `clinical-reports` (kebab-case, idéntico al de la referencia → también hunde el name exact-match de L3 a 0).
3. **En su lugar, el modelo inventó un schema «razonable»**: `requirements`, `parameters`, `outputFormat`… Es JSON con sentido común de API, no el Skill Blueprint. El modelo no está «fallando en JSON»: está **alucinando el contrato**.

Esta es la conclusión que justifica la Fase 3: el problema no es de capacidad (L1 demuestra que genera JSON completo y sobre el tema) sino de **contrato** — y los contratos exactos se enseñan con ejemplos supervisados, no con prompts. El fine-tuning QLoRA (E1) entrena con loss *solo* sobre el blueprint correcto (§8): cada ejemplo le dice al modelo «dada esta petición, el output es ESTE schema, con ESTOS campos, y termina en EOS».

Si E1 funciona, los primeros números que tienen que moverse son L2 y L5 (de 0 a algo sustancial); si L2 no se mueve, el problema estará en los datos o en el masking — no en «más epochs».

## Reproducir el SMOKE (no ejecutar aquí)

La generación ya está hecha y guardada en `experiments/e0_smoke/`; este notebook solo la analiza. Para regenerarla desde la raíz del repo (tarda minutos en CPU y requiere descargar el modelo de 0.5B):

```bash
HF_HUB_DISABLE_XET=1 uv run python scripts/run_baseline.py \
    --model Qwen/Qwen2.5-0.5B-Instruct --mode zero_shot \
    --split val --limit 5 --output-dir experiments/e0_smoke
```

*(En esta VM el descargador Xet de Hugging Face se quedó a 0 bytes; por eso `HF_HUB_DISABLE_XET=1`.)*

El **E0 real** — Qwen2.5-7B-Instruct, val y test completos, zero-shot (E0a) y few-shot (E0b) — se ejecutará en Kaggle con la misma herramienta (`scripts/run_baseline.py` sin `--limit` y con el modelo de 7B). Hasta entonces, cualquier «resultado E0» que veas citado con el 0.5B es el SMOKE, no E0.